# GIPT summary tables ? driver

Run this in Google Colab. Steps:

1. Clone or pull this repo (private ? needs a GitHub token in Colab Secrets).
2. Install requirements.
3. Mount Drive (for the source files and the service-account credentials).
4. Load the GIPT once, then update each workbook independently.


In [ ]:
# Private repo: add a GitHub token in Colab Secrets (key icon, left sidebar)
# named GITHUB_PAT, then run this cell. Clones on a fresh runtime, or pulls
# the latest if the repo is already present from earlier in this session.
import os
from google.colab import userdata

tok = userdata.get('GITHUB_PAT')
REPO = '/content/GIPT'
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://{tok}@github.com/GlobalEnergyMonitor/GIPT.git {REPO}
%cd /content/GIPT/summary_tables

# Public-repo variant (no token):
#   !git clone https://github.com/GlobalEnergyMonitor/GIPT.git {REPO}


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from src.data import load_gipt
from src.checks import summarise_gipt

data = load_gipt()        # reads GIPT + GSPT from Drive (paths in src/config.py)
summarise_gipt(data)


## Update a workbook

`build()` returns `{tab: dataframe}` so you can inspect before writing. Push
either the convenient way (`run(..., write=True)`) or tab-by-tab with full
control over the start cell and header/index flags.


In [ ]:
from src import sheets
from src.workbooks import region_by_tech as rbt

# Dry run: build all 8 tabs + print a per-tab summary; writes nothing.
frames = rbt.run(data, write=False)
frames['Operating']           # inspect any tab's dataframe before writing

# --- To push (uncomment) ---
# Convenience: all tabs at the sheet's anchor (B6):
# rbt.run(data, write=True)
#
# Tab-by-tab, full control (exact cell + header/index flags), like before:
# ss = sheets.open_sheet(rbt.SHEET_KEY)
# sheets.write_frame(ss, 'Operating', frames['Operating'], 'B6',
#                    copy_head=False, copy_index=False, extend=False, fit=False)
#
# Inspect the solar conversion (per-project AC probabilities by region):
# from src.data import solar_ac_table
# solar_ac_table().groupby('Region')['ac_probability'].describe()
